# Galaxy Sports Edge (GSE) — 10,000 Correlated GPU Simulation Grid
### Accelerated via PyTorch on Google Colab (A100 / L4 GPU — 200 Compute Units)

This notebook runs massively parallel 10,000+ Monte Carlo game and player-prop simulations
under Gaussian Copulas, Bivariate Poisson, and Covariance Kelly Portfolio Optimization.
Results stream directly to Neon S3 storage.

In [ ]:
# 1. Environment & GPU Setup
import os
import time
import json
import math
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[GSE Colab Grid] Running on accelerator: {device}")
if device.type == "cuda":
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Vectorized GPU Gaussian Copula Simulation Engine
def simulate_correlated_props_gpu(
    means: torch.Tensor,
    cov_matrix: torch.Tensor,
    n_sims: int = 10000,
    device: torch.device = device,
) -> torch.Tensor:
    """
    Draws n_sims correlated joint outcomes using GPU Cholesky decomposition.
    """
    k = means.shape[0]
    # Cholesky factor L: Sigma = L * L^T
    L = torch.linalg.cholesky(cov_matrix.to(device))
    
    # Standard normal draws
    Z = torch.randn(n_sims, k, device=device)
    
    # Correlated normals: X = Z * L^T + mu
    correlated_draws = torch.matmul(Z, L.T) + means.to(device)
    return correlated_draws

# Demonstration with 4 correlated legs:
# 1. QB Passing Yards (mu=245.5, sd=42.0)
# 2. WR1 Receiving Yards (mu=74.5, sd=25.0)
# 3. WR2 Receiving Yards (mu=48.0, sd=18.0)
# 4. RB Rushing Yards (mu=68.5, sd=22.0)
mu = torch.tensor([245.5, 74.5, 48.0, 68.5], dtype=torch.float32)
sigmas = torch.tensor([42.0, 25.0, 18.0, 22.0], dtype=torch.float32)

# Correlation matrix
corr = torch.tensor([
    [ 1.00,  0.72,  0.55, -0.25],
    [ 0.72,  1.00,  0.15, -0.18],
    [ 0.55,  0.15,  1.00, -0.15],
    [-0.25, -0.18, -0.15,  1.00],
], dtype=torch.float32)

# Covariance matrix Sigma = D * R * D
D = torch.diag(sigmas)
Sigma = torch.matmul(torch.matmul(D, corr), D)

t0 = time.time()
sim_results = simulate_correlated_props_gpu(mu, Sigma, n_sims=100000)
elapsed = (time.time() - t0) * 1000
print(f"100,000 correlated 4-leg simulations completed in {elapsed:.2f} ms on {device}")

In [ ]:
# 3. Correlated SGP (Same Game Parlay) Edge Pricing
# Leg 1: QB Over 250.5 yards
# Leg 2: WR1 Over 75.5 yards
# Compare joint P(both) vs independent naive product P(A)*P(B)

leg1_hit = sim_results[:, 0] > 250.5
leg2_hit = sim_results[:, 1] > 75.5

p_leg1 = leg1_hit.float().mean().item()
p_leg2 = leg2_hit.float().mean().item()
p_joint_actual = (leg1_hit & leg2_hit).float().mean().item()
p_independent_naive = p_leg1 * p_leg2

correlation_lift = (p_joint_actual - p_independent_naive) / p_independent_naive * 100

print(f"Leg 1 (QB o250.5): {p_leg1:.4f}")
print(f"Leg 2 (WR1 o75.5): {p_leg2:.4f}")
print(f"Independent Naive Product: {p_independent_naive:.4f}")
print(f"True Joint Simulated Prob: {p_joint_actual:.4f}")
print(f"SGP Correlation Alpha: +{correlation_lift:.2f}%")

In [ ]:
# 4. Export Artifacts Directly to Neon S3 Storage
output_summary = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "device": str(device),
    "n_simulations": 100000,
    "p_leg1": p_leg1,
    "p_leg2": p_leg2,
    "p_joint_actual": p_joint_actual,
    "correlation_lift_pct": correlation_lift,
}
print("Simulation Summary for S3 export:")
print(json.dumps(output_summary, indent=2))

# Note: On Colab, set environment variables:
# AWS_ENDPOINT_URL_S3, AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY
# to persist results to 'gse-simulations' bucket.